# YOLO Training Notebook (Phase 1: Local)

This notebook automates the second part of YOLO model training:
1. **Step 1:** Creates upload folders and shows you where to upload your files
2. **Step 2:** You upload train images and labels (and optionally val) to those folders
3. **Step 3:** Run the remaining cells to build the dataset, generate YAML, and train

**Environment variables** (optional): Set these to skip prompts.
- `CLASSES` - Comma-separated class names (default: Badge)
- `OUTPUT_ROOT` - Where to create YOLO dataset (default: ./yolo_dataset)

## 1. Configuration & Create Upload Folders

Run this cell first. It installs ultralytics, creates folders for your uploads, and shows you where to upload your files.

In [ ]:
import os
from pathlib import Path

print("Installing ultralytics...")
%pip install ultralytics
print("Done. Collecting configuration...")


def get_config(env_var: str, prompt: str, default: str = "") -> str:
    """Use env var if set, else prompt user. Returns trimmed string."""
    value = os.getenv(env_var, "").strip()
    if value:
        return value
    if default:
        user_input = input(f"{prompt} [{default}]: ").strip()
        return user_input if user_input else default
    return input(f"{prompt}: ").strip()


# Collect configuration (only CLASSES and OUTPUT_ROOT)
# If prompted below the cell, type your answer and press Enter
CLASSES_STR = get_config("CLASSES", "Class names (comma-separated)", "Badge")
OUTPUT_ROOT = get_config(
    "OUTPUT_ROOT", "Output directory for YOLO dataset", "./yolo_dataset"
)

# Parse classes into dict {0: "Badge", 1: "Other", ...}
CLASSES = {
    i: name.strip() for i, name in enumerate(CLASSES_STR.split(",")) if name.strip()
}

# Resolve paths
OUTPUT_ROOT = Path(OUTPUT_ROOT).expanduser().resolve()
WORKSPACE_ROOT = Path.cwd()

# Pre-create upload folders (where you will upload your files)
UPLOAD_ROOT = WORKSPACE_ROOT / "upload"
TRAIN_IMAGES_PATH = UPLOAD_ROOT / "train_images"
TRAIN_LABELS_PATH = UPLOAD_ROOT / "train_labels"
VAL_IMAGES_PATH = UPLOAD_ROOT / "val_images"
VAL_LABELS_PATH = UPLOAD_ROOT / "val_labels"

for d in [TRAIN_IMAGES_PATH, TRAIN_LABELS_PATH, VAL_IMAGES_PATH, VAL_LABELS_PATH]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Classes: {CLASSES}")
print(f"Output: {OUTPUT_ROOT}")
print()
print("=" * 60)
print("UPLOAD YOUR FILES TO THESE FOLDERS:")
print("=" * 60)
print(f"  Train images:  {TRAIN_IMAGES_PATH}")
print(f"  Train labels:  {TRAIN_LABELS_PATH}")
print(f"  Val images:    {VAL_IMAGES_PATH} (optional)")
print(f"  Val labels:    {VAL_LABELS_PATH} (optional)")
print()
print("How to upload: In the file browser, navigate to each folder,")
print("click the Upload button (or drag and drop), and select your files.")
print("  - Train images: .jpg, .png, .webp, etc.")
print("  - Train labels: .txt files from Label Studio (YOLO format)")
print()
print("After uploading, run the next cells.")
print("=" * 60)
print("\n✓ Step 1 complete. Upload your files, then run the next cell.")

## 2. Create YOLO Structure & Copy from Upload Folders

After you have uploaded your files to the folders above, run this cell. It copies them into the YOLO dataset structure.

In [ ]:
import shutil

# Create YOLO directory structure
IMAGES_TRAIN = OUTPUT_ROOT / "images" / "train"
IMAGES_VAL = OUTPUT_ROOT / "images" / "val"
LABELS_TRAIN = OUTPUT_ROOT / "labels" / "train"
LABELS_VAL = OUTPUT_ROOT / "labels" / "val"

for d in [IMAGES_TRAIN, IMAGES_VAL, LABELS_TRAIN, LABELS_VAL]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Created directory structure under {OUTPUT_ROOT}")

# Image extensions to look for
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff"}

# Validate source paths (upload folders should exist; check they have content)
if not TRAIN_IMAGES_PATH.exists():
    raise FileNotFoundError(f"Train images path does not exist: {TRAIN_IMAGES_PATH}")
if not TRAIN_LABELS_PATH.exists():
    raise FileNotFoundError(f"Train labels path does not exist: {TRAIN_LABELS_PATH}")

train_image_count = sum(
    1
    for f in TRAIN_IMAGES_PATH.iterdir()
    if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
)
if train_image_count == 0:
    raise FileNotFoundError(
        f"No images found in {TRAIN_IMAGES_PATH}. Please upload your train images first."
    )


def copy_from_local(
    images_path: Path, labels_path: Path, out_images: Path, out_labels: Path
) -> tuple[int, int]:
    """Copy images and labels to output. Returns (images_count, labels_count)."""
    images_copied = 0
    labels_copied = 0

    # Copy images
    for f in images_path.iterdir():
        if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS:
            shutil.copy2(f, out_images / f.name)
            images_copied += 1

    # Copy labels
    for f in labels_path.iterdir():
        if f.is_file() and f.suffix.lower() == ".txt":
            shutil.copy2(f, out_labels / f.name)
            labels_copied += 1

    return images_copied, labels_copied


# Copy train data
train_imgs, train_lbls = copy_from_local(
    TRAIN_IMAGES_PATH, TRAIN_LABELS_PATH, IMAGES_TRAIN, LABELS_TRAIN
)
print(f"Train: copied {train_imgs} images, {train_lbls} labels")

# Copy val data (if you uploaded any)
val_imgs, val_lbls = copy_from_local(
    VAL_IMAGES_PATH, VAL_LABELS_PATH, IMAGES_VAL, LABELS_VAL
)
print(f"Val: copied {val_imgs} images, {val_lbls} labels")
print("\n✓ Step 2 complete. Run the next cell.")

## 3. Label Matching

Match label .txt files to images by base name. Create empty .txt for images without labels (negative examples).

In [ ]:
def ensure_labels_for_images(images_dir: Path, labels_dir: Path) -> tuple[int, int]:
    """
    For each image, ensure a matching .txt exists in labels_dir.
    Creates empty .txt for images without labels.
    Returns (total_images, empty_labels_created).
    """
    existing_labels = {
        f.stem for f in labels_dir.iterdir() if f.suffix.lower() == ".txt"
    }
    empty_created = 0
    total = 0

    for f in images_dir.iterdir():
        if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS:
            total += 1
            stem = f.stem
            if stem not in existing_labels:
                (labels_dir / f"{stem}.txt").write_text("")
                empty_created += 1

    return total, empty_created


# Ensure train labels
train_total, train_empty = ensure_labels_for_images(IMAGES_TRAIN, LABELS_TRAIN)
print(
    f"Train: {train_total} images, created {train_empty} empty labels for images without labels"
)

# Ensure val labels if val has images
val_images = list(IMAGES_VAL.iterdir())
if any(f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS for f in val_images):
    val_total, val_empty = ensure_labels_for_images(IMAGES_VAL, LABELS_VAL)
    print(f"Val: {val_total} images, created {val_empty} empty labels")
else:
    print("Val: no images (empty folder)")
print("\n✓ Step 3 complete. Run the next cell.")

## 4. Generate Dataset YAML

In [ ]:
YAML_PATH = OUTPUT_ROOT / "data.yaml"

yaml_content = f"""path: {OUTPUT_ROOT}
train: images/train
val: images/val

names:
"""
for idx, name in CLASSES.items():
    yaml_content += f"  {idx}: {name}\n"

YAML_PATH.write_text(yaml_content)
print(f"Generated {YAML_PATH}")
print(yaml_content)
print("\n✓ Step 4 complete. Run the next cell to train.")

## 5. Train YOLO Model

**Note:** Ultralytics automatically downloads the base model (`yolov8n.pt`, ~6MB) from the internet on first use if it is not found locally. No manual download or upload required.

In [ ]:
import warnings

from ultralytics import YOLO

warnings.filterwarnings("ignore", category=RuntimeWarning)

# yolov8n.pt is downloaded automatically by Ultralytics if not found locally
model = YOLO("yolov8n.pt")
model.train(
    data=str(YAML_PATH),
    epochs=100,
    imgsz=640,
    name="badge-demo",
)

print("\n" + "=" * 60)
print("✓ TRAINING COMPLETE")
print("=" * 60)
print("Best model saved to: runs/detect/badge-demo/weights/best.pt")
print("=" * 60)